# State Observation

<div class="alert alert-success">In this notebook, I implemented the function `data_to_observations()` to a 60-day lookback data for each observations in rolling window. This adds a temporal dimension to the input. Providing only a single data point to a Reinforcement Learning (RL) agent is insufficient because a single day-event lacks the context of change or trend. As noted in resources like the HuggingFace Deep RL course, environments like Atari DQN stack 3–4 consecutive frames to capture velocity and direction. Stacking and grouping historical days allows the model to perceive the context for the observation. </div> 

In [1]:
# using the same data loading technique. 
import pandas as pd
# load the parquet file data into pandas dataframe
def load_parquet(file_path: str) -> pd.DataFrame:
    try:
        data = pd.read_parquet(file_path, engine='pyarrow')
        return data
    except Exception as e:
        print(f"An error occurred while importing data: {e}")
        return pd.DataFrame()


In [2]:
# Continuing from the last notebook, where I loaded the file and it had notmalized data.
processed_aapl = pd.read_parquet('train_dataset/AAPL_processed.parquet', engine='pyarrow')
display(processed_aapl.head())

,Close,RSI_norm,MACD_line_norm,MACD_signal_norm,MACD_diff_norm,Volume_norm,Return,Target
Date,,,,,,,,
1982-02-10,0.334821,-0.168715,0.052043,0.265932,-0.562678,0.536293,0.013423,0.294643
1982-02-11,0.330357,-0.203611,-0.014915,0.209773,-0.606959,-0.166255,-0.013423,0.290179
1982-02-12,0.334821,-0.151552,-0.039855,0.159323,-0.547999,-0.507200,0.013423,0.296875
1982-02-16,0.328125,-0.206961,-0.092100,0.107611,-0.565850,0.349158,-0.020203,0.301339
1982-02-17,0.332589,-0.152905,-0.103854,0.063683,-0.483898,-0.103271,0.013514,0.301339


The dataset is now prepared for windowing. I will reshape the input into discrete samples, where each sequence represents a 60-day lookback window incorporating the normalized RSI, MACD (line and signal), Volume, and Return features.

In [ ]:

import numpy as np
from tqdm import tqdm

def data_to_observations(data: pd.DataFrame, window: int = 60) -> tuple[np.ndarray, np.ndarray]:
    features = ['RSI_norm', 'MACD_line_norm', 'MACD_signal_norm', 'Volume_norm', 'Return']
    X = []
    y = []
    
    for i in tqdm(range(len(data) - window)):
        X.append(data[features].iloc[i:i+window].values)
        # The value should be the 60th period values as I am wrapping up previous 60 days of data.
        # Corrected below to 1 less index to select the correct data
        y.append({
            'target': data['Target'].iloc[i+window-1], 
            'date': data.index[i+window-1], 
            'close': data['Close'].iloc[i+window-1]})
    return np.array(X), np.array(y)

In [30]:
x_apple, y_apple = data_to_observations(processed_aapl)

100%|██████████| 9557/9557 [00:03<00:00, 2936.76it/s]


In [ ]:
display("🍀",x_apple.shape)
display("🍀",y_apple.shape)

(9557, 60, 5)

(9557,)

In [32]:
display(y_apple[0])

{'target': np.float64(0.2299107164144516),
 'date': Timestamp('1982-05-06 00:00:00'),
 'close': np.float64(0.2857142984867096)}

In [33]:
processed_aapl_val = pd.read_parquet('validation_dataset/AAPL_processed.parquet', engine='pyarrow')
display(processed_aapl_val.head())

,Close,RSI_norm,MACD_line_norm,MACD_signal_norm,MACD_diff_norm,Volume_norm,Return,Target
Date,,,,,,,,
2021-06-01,124.279999,-0.162555,-1.131932,-1.185974,-0.128403,-1.502697,-0.002652,149.149994
2021-06-02,125.059998,-0.112683,-1.118775,-1.190792,-0.076224,-1.843607,0.006257,148.479996
2021-06-03,123.540001,-0.185968,-1.152905,-1.202003,-0.146132,-1.156407,-0.012229,146.389999
2021-06-04,125.889999,-0.042596,-1.092184,-1.197659,0.021889,-1.192238,0.018844,142.449997
2021-06-07,125.900002,-0.042018,-1.037881,-1.182311,0.141948,-1.333705,0.000079,146.149994


In [34]:
x_val_apple, y_val_apple = data_to_observations(processed_aapl_val)

100%|██████████| 1113/1113 [00:00<00:00, 3297.68it/s]


In [47]:
display("\U0001F453 Values of the new observation: ", y_val_apple[0], x_val_apple[0][59])

'👓 Values of the new observation: '

{'target': np.float64(142.0),
 'date': Timestamp('2021-08-24 00:00:00'),
 'close': np.float64(149.6199951171875)}

array([ 1.95432034e-01,  1.36967040e-01,  2.08152544e-01, -1.95938837e+00,
       -6.01420498e-04])

In [46]:
display("\U0001F453 Values of the original data: ", processed_aapl_val.loc[y_val_apple[0]['date']])

'👓 Values of the original data: '

Close               149.619995
RSI_norm              0.195432
MACD_line_norm        0.136967
MACD_signal_norm      0.208153
MACD_diff_norm       -0.164183
Volume_norm          -1.959388
Return               -0.000601
Target              142.000000
Name: 2021-08-24 00:00:00, dtype: float64

In [50]:
print("\n🌱 Length of the observation: ", len(x_val_apple[0]))
print("\n🌱 Shape of the original data: ", processed_aapl_val.shape)
print("\n🌱 Shape of the observation: ", x_val_apple.shape)



🌱 Length of the observation:  60

🌱 Shape of the original data:  (1173, 8)

🌱 Shape of the observation:  (1113, 60, 5)


## Conclusion
🍋 Implemented the code to convert the data shape in (n, 60, 5) for the RL model training.   
🍋 Ready for training RL now.  


|Date (YYYY-MM-DD)|Version|Created By|  
|--|--|--|
|2026-02-27|1.0|Battogtokh Baasanjav|